# IDA CFG Analysis with ipysigma

This notebook demonstrates how to extract, load, and visualize the Control Flow Graph (CFG) from an IDA database.


### 1. Extract CFG from IDA Database
This step requires `idapro` 9.0+ and a valid license.


In [11]:
import sys
import os
from src.extract_cfg import extract_cfg_from_db
import os

# Replace with the path to your IDA database (.idb or .i64)
db_path = "/Users/mark/windows_share/test/reorder_and_pad.exe.i64"
json_path = "reorder_and_pad2.json"

if os.path.exists(db_path):
    json_path = extract_cfg_from_db(db_path, output_path=json_path)
else:
    print(f"Database {db_path} not found. Skipping extraction.")


Opening database: /Users/mark/windows_share/test/reorder_and_pad.exe.i64
Attempting to open database...
Database opened with handle: 0
Number of functions found: 343
Extracting functions and intra-function edges...
Found 343 functions.
Extracting inter-function calls...
CFG exported to reorder_and_pad2.json
Closing database...


### 2. Load and Visualize the Graph with ipysigma


In [12]:
import importlib
importlib.invalidate_caches()
from ipysigma import Sigma
import pandas as pd
import networkx as nx
import src.visualize_cfg as cfg
importlib.reload(cfg)
import os

# Load the graph data
G = cfg.load_cfg(json_path)

if G is not None:
    print(f"Graph loaded: {len(G.nodes)} nodes, {len(G.edges)} edges")
    to_be_removed = [x for  x in G.nodes() if G.degree(x) <= 1]
    G.remove_nodes_from(to_be_removed)
    # Basic info
    print(f"Number of functions after removing degree <= 1: {len(G.nodes)}")

    # Collapse chains (nodes with 1 caller and 1 callee)
    G = cfg.collapse_chains(G)
    print(f"Number of nodes after collapsing chains: {len(G.nodes)}")


    # Visualize with ipysigma
    widget = Sigma(
        G,
        node_label='label',
        node_color='color',
        node_size=lambda n, d: 10 if d.get('func') != "unknown" else 5,
        edge_color='type',
        label_font='sans-serif',
        default_edge_type='arrow'
    )
    display(widget)

else:
    print(f"Could not load graph from {json_path}. Please make sure the file exists and it is a valid CFG JSON.")


Graph loaded: 6021 nodes, 8676 edges
Number of functions after removing degree <= 1: 5014
Number of nodes after collapsing chains: 3312


Sigma(nx.DiGraph with 3,312 nodes and 5,640 edges)

Ways to reduce:
- strongly connected components
- remove nodes with degree <= 1
- clique detection and collapse
- graph community collapse